In [ ]:
import os
import time
import random
from collections import Counter
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import faiss
from github import Github
import google.generativeai as genai
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

GITHUB_TOKEN = os.getenv("GITHUB_PAT")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GITHUB_TOKEN or not GEMINI_API_KEY:
    print("Warning: API keys not found in .env file.")
else:
    g = Github(GITHUB_TOKEN)
    genai.configure(api_key=GEMINI_API_KEY)
    print(f"Authenticated GitHub user: {g.get_user().login}")

In [ ]:
def scrape_commits(query_list, commits_per_query=10):
    extracted_data = []
    
    for query in query_list:
        print(f"Searching for: '{query}'...")
        try:
            commits = g.search_commits(query=query)
            count = 0
            for commit in commits:
                if count >= commits_per_query:
                    break
                try:
                    msg = commit.commit.message.split('\n')[0]
                    author = commit.commit.author.name if commit.commit.author else "Unknown"
                    repo = commit.repository.full_name
                    url = commit.html_url

                    extracted_data.append({
                        "author": author,
                        "repo": repo,
                        "commit_message": msg,  # Explicitly named to avoid KeyErrors
                        "url": url
                    })
                    count += 1
                    time.sleep(0.5) 
                except Exception:
                    continue
        except Exception as e:
            print(f"API paused on '{query}': {e}. Resting...")
            time.sleep(5)
            continue

    df = pd.DataFrame(extracted_data)
    print(f"Extraction complete. Total commits: {len(df)}")
    return df

queries = [
    "please work", "i have no idea why this works", 
    "this is terrible", "fixed it again", 
    "i give up", "spaghetti code"
]

df_commits = scrape_commits(query_list=queries, commits_per_query=10)
df_commits.head()

In [ ]:
# Keep only commit messages and remove duplicates
df = df_commits.copy()
df = df[["commit_message"]].drop_duplicates()

stress_keywords = [
    "please", "again", "error", "bug", "terrible", "give up",
    "stupid", "why", "spaghetti", "not working", "doesnt work"
]

def label_commit(text):
    text = text.lower()
    return 1 if any(word in text for word in stress_keywords) else 0

df["label"] = df["commit_message"].apply(label_commit)

# Synthetic data templates
frustration_templates = [
    "fixing {} again", "why is this {} happening", 
    "debugging this {} for hours", "this {} still not working",
    "another {} error", "trying to fix {} again", "this {} is terrible"
]
bug_words = ["bug", "error", "issue", "problem", "failure", "glitch"]

normal_templates = [
    "added {} feature", "updated {} module", "improved {} performance",
    "refactored {} code", "optimized {} query", "cleaned {} implementation"
]
normal_words = ["login", "authentication", "database", "UI", "API", "system"]

# Generate synthetic commits
synthetic_commits = []
for _ in range(400):
    synthetic_commits.append(random.choice(frustration_templates).format(random.choice(bug_words)))
for _ in range(200):
    synthetic_commits.append(random.choice(normal_templates).format(random.choice(normal_words)))

synthetic_df = pd.DataFrame({"commit_message": synthetic_commits})
synthetic_df["label"] = synthetic_df["commit_message"].apply(label_commit)

# Merge and shuffle dataset
final_df = pd.concat([df, synthetic_df], ignore_index=True)
final_df = final_df.sample(frac=1).reset_index(drop=True)

print(f"Original commits: {len(df)}")
print(f"Final dataset size: {len(final_df)}")
final_df.head()

In [ ]:
texts = final_df["commit_message"].tolist()
labels = final_df["label"].tolist()

# Build vocabulary
all_words = [word for text in texts for word in text.lower().split()]
word_counts = Counter(all_words)
vocab = {word: i+1 for i, (word, _) in enumerate(word_counts.items())}
vocab_size = len(vocab) + 1

print(f"Vocabulary size: {vocab_size}")

def text_to_sequence(text):
    return [vocab.get(word, 0) for word in text.lower().split()]

sequences = [text_to_sequence(text) for text in texts]
max_len = 10

def pad_sequence(seq):
    if len(seq) < max_len:
        return seq + [0] * (max_len - len(seq))
    return seq[:max_len]

padded_sequences = [pad_sequence(seq) for seq in sequences]

# Convert to tensors
X = torch.tensor(padded_sequences)
y = torch.tensor(labels)

print(f"Tensor shape (X): {X.shape}")
print(f"Tensor shape (y): {y.shape}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Prepare train/test splits (This was missing in your original code!)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

class LSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.W_i = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_f = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_c = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_o = nn.Linear(input_size + hidden_size, hidden_size)

    def forward(self, x, h_prev, c_prev):
        combined = torch.cat((x, h_prev), dim=1)
        i = torch.sigmoid(self.W_i(combined))
        f = torch.sigmoid(self.W_f(combined))
        o = torch.sigmoid(self.W_o(combined))
        c_tilde = torch.tanh(self.W_c(combined))
        
        c = f * c_prev + i * c_tilde
        h = o * torch.tanh(c)
        return h, c

class CustomBiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.hidden_dim = hidden_dim
        self.lstm_forward = LSTMCell(embed_dim, hidden_dim)
        self.lstm_backward = LSTMCell(embed_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        x = self.embedding(x)
        batch_size, seq_len, _ = x.shape
        
        h_f = torch.zeros(batch_size, self.hidden_dim)
        c_f = torch.zeros(batch_size, self.hidden_dim)
        h_b = torch.zeros(batch_size, self.hidden_dim)
        c_b = torch.zeros(batch_size, self.hidden_dim)

        forward_outputs, backward_outputs = [], []

        for t in range(seq_len):
            h_f, c_f = self.lstm_forward(x[:, t, :], h_f, c_f)
            forward_outputs.append(h_f)

        for t in reversed(range(seq_len)):
            h_b, c_b = self.lstm_backward(x[:, t, :], h_b, c_b)
            backward_outputs.append(h_b)

        h_concat = torch.cat((forward_outputs[-1], backward_outputs[-1]), dim=1)
        return torch.sigmoid(self.fc(h_concat))

# Initialize and train
model = CustomBiLSTM(vocab_size)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 10
for epoch in range(epochs):
    outputs = model(X_train)
    loss = criterion(outputs.squeeze(), y_train.float())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

# Evaluate
model.eval()
with torch.no_grad():
    predictions = model(X_test).squeeze()
    preds = (predictions > 0.5).int()
    print(f"\nTest Accuracy: {accuracy_score(y_test, preds):.4f}")

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# Load embedding model and generate vectors
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
commit_texts = final_df["commit_message"].tolist()
embeddings = embed_model.encode(commit_texts)

# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print(f"FAISS index built successfully. Total vectors stored: {index.ntotal}")

def get_similar_commits(query, k=5):
    query_vector = embed_model.encode([query])
    distances, indices = index.search(query_vector, k)
    results = []
    for i in indices[0]:
        if i < len(final_df):
            results.append(final_df.iloc[i]["commit_message"])
    return results

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Confusion Matrix
cm = confusion_matrix(y_test, preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
            xticklabels=["Normal", "Frustrated"], 
            yticklabels=["Normal", "Frustrated"], ax=axes[0])
axes[0].set_title("Commit Sentiment Confusion Matrix")
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Label")

# Plot 2: Dataset Distribution
counts = final_df["label"].value_counts()
axes[1].bar(["Normal Commits", "Frustrated Commits"], counts.values, color=['#4C72B0', '#DD8452'])
axes[1].set_title("Commit Emotion Distribution")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
def generate_hr_mail(commit):
    subjects = [
        "Another Day, Another Bug", "Production is on Fire Again",
        "The Bug That Refuses to Die", "Daily Dose of Debugging Pain"
    ]
    openings = [
        "After several heroic attempts", "Following hours of intense debugging",
        "In a shocking turn of events", "Despite our best efforts"
    ]
    closings = [
        "Please join me in pretending we understand why this works.",
        "The code remains mysterious but functional.",
        "Further sacrifices to the debugging gods may be required.",
        "We will continue investigating this technological marvel."
    ]
    
    mail = f"""Subject: {random.choice(subjects)}

Dear Team,

{random.choice(openings)}, the following commit was made:
"{commit}"

Naturally, the system continues to behave in ways that defy both logic and documentation.

{random.choice(closings)}

Best regards,
A slightly tired developer
"""
    return mail

def predict_frustration(commit):
    words = commit.lower().split()
    seq = [vocab.get(w, 0) for w in words]
    seq = seq + [0] * (max_len - len(seq)) if len(seq) < max_len else seq[:max_len]
    
    tensor = torch.tensor([seq]).long()
    model.eval()
    
    with torch.no_grad():
        output = model(tensor)
        prob = float(output.squeeze().cpu().numpy())
    return prob

In [ ]:
import gradio as gr

def analyze_commit_ui(commit):
    try:
        score = predict_frustration(commit)
        similar = get_similar_commits(commit)
        similar_text = "\n".join([f"- {s}" for s in similar])
        percent = int(score * 100)

        if score > 0.5:
            status = f"Developer Frustration Detected ({percent}%)"
            mail = generate_hr_mail(commit)
        else:
            status = f"Normal Development Activity ({percent}%)"
            mail = "Subject: Calm and Professional Commit\n\nDear Team,\n\nIt appears that this commit was written in a calm and composed state.\n\nNo debugging-induced emotional meltdown detected.\n\nBest regards,\nAI Commit Analyzer"

        return status, percent, similar_text, mail
    except Exception as e:
        return f"Error: {str(e)}", 0, "Error retrieving commits", "Mail generation failed"

# UI Layout
with gr.Blocks() as demo:
    gr.Markdown("## Developer Commit AI Analyzer\nDetect developer frustration from commit messages using **BiLSTM + Sentence-BERT + FAISS**.")
    
    commit_input = gr.Textbox(label="Enter Commit Message", placeholder="e.g., why is this bug still happening", lines=2)
    analyze_btn = gr.Button("Analyze Commit")
    
    with gr.Row():
        status_box = gr.Textbox(label="AI Sentiment Analysis Result")
        rage_meter = gr.Slider(minimum=0, maximum=100, label="Developer Frustration Level (%)")
        
    similar_box = gr.Textbox(label="Similar Commits Retrieved", lines=4)
    mail_box = gr.Textbox(label="Generated Sarcastic Developer Mail", lines=10)

    analyze_btn.click(
        analyze_commit_ui,
        inputs=commit_input,
        outputs=[status_box, rage_meter, similar_box, mail_box]
    )

# Note: Added share=True so you can easily send the link during an interview!
demo.launch(share=True)